# 03 - torch.distributed 简单 Demo

> 配套笔记：[01. 分布式训练(0) - 背景知识](https://github.com/Zoey-Cheng/MLSys-Learning-Notes/blob/main/notes/02_训练策略/02_01_分布式训练基础.md) ｜ 知乎：<https://zhuanlan.zhihu.com/p/1897578451143221835>

两部分：
1. **通信原语**：broadcast / reduce / all_reduce / gather / all_gather / scatter / reduce_scatter / barrier
2. **自定义分布式训练**：DDP + DistributedSampler 的 minimal demo

> ⚠️ Colab 免费版只给 1 GPU，所以这里用 **gloo 后端 + CPU 模拟**。生产用 **NCCL + GPU**。

## 1. 通信原语

`torch.distributed` 标准流程 6 步：初始化进程组 → (子组) → (DDP) → (Sampler) → 启动器 → 销毁。

通信原语测试只用到 (1) 初始化和最后的销毁，所以把它们封装成 `setup` / `cleanup`。

### 1.1 setup / cleanup

复用的 boilerplate，写到 `dist_utils.py`。

In [ ]:
%%writefile dist_utils.py
import os
import torch
import torch.distributed as dist

def setup(rank, world_size, port=12355):
    dist.init_process_group(
        backend='gloo',
        init_method=f'tcp://127.0.0.1:{port}',
        rank=rank,
        world_size=world_size
    )

def cleanup():
    dist.destroy_process_group()


### 1.2 各原语示例

每个原语一个函数。两个细节：
- 用 `log()` 包一层 `print(..., flush=True)`，避免 spawn/join 卡死
- 每个函数末尾加 `dist.barrier()`，避免结束时锁死

In [ ]:
%%writefile commu_funcs.py
import torch
import torch.distributed as dist
import torch.multiprocessing as mp
import time
import sys
from dist_utils import setup, cleanup

# ------------------- 打印封装 ----------------------
def log(msg, rank=None):
    prefix = f"[Rank {rank}] " if rank is not None else ""
    print(f"{prefix}{msg}", flush=True)

# ------------------- 通信原语 ----------------------

def example_broadcast(rank):
    tensor = torch.tensor([1, 2, 3, 4, 5], dtype=torch.float32) if rank == 0 else torch.zeros(5)
    log(f"Before Broadcast: {tensor}", rank)
    dist.broadcast(tensor, src=0)
    log(f"After Broadcast: {tensor}", rank)
    dist.barrier()

def example_reduce(rank):
    tensor = torch.tensor([rank + 1] * 5, dtype=torch.float32)
    log(f"Before Reduce: {tensor}", rank)
    dist.reduce(tensor, dst=0, op=dist.ReduceOp.SUM)
    log(f"After Reduce: {tensor}", rank)
    dist.barrier()

def example_all_reduce(rank):
    tensor = torch.tensor([rank + 1] * 5, dtype=torch.float32)
    log(f"Before AllReduce: {tensor}", rank)
    dist.all_reduce(tensor, op=dist.ReduceOp.SUM)
    log(f"After AllReduce: {tensor}", rank)
    dist.barrier()

def example_gather(rank):
    tensor = torch.tensor([rank + 1] * 5, dtype=torch.float32)
    gather_list = [torch.zeros(5) for _ in range(dist.get_world_size())] if rank == 0 else None
    log(f"Before Gather: {tensor}", rank)
    dist.gather(tensor, gather_list, dst=0)
    if rank == 0:
        log(f"After Gather: {gather_list}", rank)
    dist.barrier()

def example_all_gather(rank):
    tensor = torch.tensor([rank + 1] * 5, dtype=torch.float32)
    gather_list = [torch.zeros(5) for _ in range(dist.get_world_size())]
    log(f"Before AllGather: {tensor}", rank)
    dist.all_gather(gather_list, tensor)
    log(f"After AllGather: {gather_list}", rank)
    dist.barrier()

def example_scatter(rank):
    if rank == 0:
        scatter_list = [torch.tensor([i + 1] * 5, dtype=torch.float32) for i in range(dist.get_world_size())]
        log(f"Rank 0 Scatter List: {scatter_list}", rank)
    else:
        scatter_list = None
    tensor = torch.zeros(5)
    log(f"Before Scatter: {tensor}", rank)
    dist.scatter(tensor, scatter_list, src=0)
    log(f"After Scatter: {tensor}", rank)
    dist.barrier()

def example_reduce_scatter(rank):
    world_size = dist.get_world_size()
    input_list = [torch.tensor([(rank + 1) * (j+1)] * 2, dtype=torch.float32) for j in range(world_size)]
    output_tensor = torch.zeros(2)
    log(f"Before ReduceScatter: {input_list}", rank)
    dist.reduce_scatter(output_tensor, input_list, op=dist.ReduceOp.SUM)
    log(f"After ReduceScatter: {output_tensor}", rank)
    dist.barrier()

def example_barrier(rank):
    log(f"Sleeping {rank} seconds...", rank)
    time.sleep(rank)
    dist.barrier()
    log(f"Passed barrier.", rank)

# ------------------- 启动器 ----------------------

def run(rank, world_size, op_name):
    setup(rank, world_size)

    try:
        ops = {
            "broadcast": example_broadcast,
            "reduce": example_reduce,
            "all_reduce": example_all_reduce,
            "gather": example_gather,
            "all_gather": example_all_gather,
            "scatter": example_scatter,
            "reduce_scatter": example_reduce_scatter,
            "barrier": example_barrier
        }

        if op_name not in ops:
            if rank == 0:
                print(f"[Rank {rank}] Unknown operation: {op_name}", flush=True)
        else:
            ops[op_name](rank)

    except Exception as e:
        log(f"Exception: {e}", rank)

    finally:
        try:
            cleanup()
        except Exception as e:
            log(f"Cleanup failed: {e}", rank)
        sys.exit(0)  # 显式退出

# ------------------- 可执行入口 ----------------------

if __name__ == "__main__":
    import sys
    op = sys.argv[1] if len(sys.argv) > 1 else "broadcast"
    world_size = 3
    mp.spawn(run, args=(world_size, op), nprocs=world_size, join=True)

### 1.3 测试执行

跑之前先确认端口空闲（上一次没释放干净的话手动 kill）。

In [ ]:
!lsof -i :12355

In [ ]:
#!kill -9 31305

In [ ]:
!python commu_funcs.py broadcast

In [ ]:
!python commu_funcs.py reduce

In [ ]:
!python commu_funcs.py all_reduce

In [ ]:
!python commu_funcs.py gather

In [ ]:
!python commu_funcs.py all_gather

In [ ]:
!python commu_funcs.py scatter

In [ ]:
!python commu_funcs.py reduce_scatter

In [ ]:
!python commu_funcs.py barrier

## 2. 自定义分布式训练

完整走通 6 步：
- (1) 初始化进程组
- (3) 模型 + DDP 封装
- (4) Dataset + DistributedSampler
- (5) `mp.spawn()` 启动（生产用 `torchrun`，调试用 spawn 更灵活）
- (6) 销毁进程组

### 2.1 工具函数

模型 / 数据集 / setup / cleanup 写到 `ddp_utils.py`。

In [ ]:
%%writefile ddp_utils.py

import torch
import torch.nn as nn
import torch.optim as optim
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler

# ------------------ 强制 CPU 模式 ------------------
torch.cuda.is_available = lambda: False

# ------------------ 1. 通信初始化 ------------------
def setup(rank, world_size, method="tcp://localhost:12355"):
    dist.init_process_group(
        backend="gloo",
        init_method=method,
        rank=rank,
        world_size=world_size
    )

# ------------------ 3. 模型 ------------------
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Linear(10, 2)
    def forward(self, x):
        return self.layer(x)

# ------------------ 4. 数据集 & Sampler ------------------
class DummyDataset(Dataset):
    def __len__(self):
        return 100
    def __getitem__(self, idx):
        return torch.randn(10), torch.randint(0, 2, (1,)).item()

def get_sampler(dataset, world_size, rank):
    return DistributedSampler(dataset, num_replicas=world_size, rank=rank)

# ------------------ 6. 销毁进程组 ------------------
def cleanup():
    dist.destroy_process_group()

### 2.2 训练主体

`train(rank, ...)` 是每个 worker 的入口，由 `mp.spawn` 拉起 `world_size` 个进程并发执行。

In [ ]:
%%writefile ddp_simple.py

import torch
import torch.nn as nn
import torch.optim as optim
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader

from ddp_utils import setup, cleanup, SimpleModel, DummyDataset, get_sampler

################### 分布式训练定义 ###################
def train(rank, world_size, epochs):
    # 1. 初始化进程组
    setup(rank, world_size)
    # 设置设备
    device = "cpu"
    print(f"[Rank {rank}] initialized", flush=True)

    # 3. 创建模型 + DDP封装
    model = SimpleModel().to(device)
    model = DDP(model)

    # 4. 数据加载器 + 分布式采样器 Sampler
    dataset = DummyDataset()
    sampler = get_sampler(dataset, world_size, rank)
    dataloader = DataLoader(dataset, batch_size=8, sampler=sampler)

    # 优化器
    optimizer = optim.SGD(model.parameters(), lr=0.01)

    # 简单训练循环（2个epoch）
    for epoch in range(epochs):
        sampler.set_epoch(epoch)
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            output = model(x)
            loss = nn.CrossEntropyLoss()(output, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        if rank == 0:
            print(f"[Rank {rank}] Epoch {epoch}, Loss: {loss.item():.4f}", flush=True)

    # 6. 销毁进程组
    cleanup()

################### 函数入口 ###################
# 5. 启动入口
if __name__ == "__main__":
    world_size = 2
    epochs = 2
    mp.spawn(train, args=(world_size, epochs), nprocs=world_size, join=True)

### 2.3 运行

In [ ]:
!python ddp_simple.py

### 流程示意图

        +---------------------+
        | torch.distributed   |
        |       launch        |
        +----------+----------+
                   |
                   v
        +---------------------+
        | 进程 0 (rank=0)     |
        |  - 初始化进程组     |
        |  - 训练模型        |
        +---------------------+
                   |
                   +--- 同步梯度 ---+
                                       |
        +---------------------+       |
        | 进程 1 (rank=1)     | <-----+
        |  - 初始化进程组     |
        |  - 训练模型        |
        +---------------------+